In [ ]:
from langchain_core.documents import Document

fake_docs = [
    Document(page_content="Amber uses CUDA for GPU acceleration in molecular dynamics simulations."),
    Document(page_content="Time-series anomaly detection in Amber can be done using trajectory analysis tools."),
    Document(page_content="Amber supports multi-tenant indexing via separate topology files."),
    Document(page_content="To troubleshoot installation errors, check your environment variables and compiler versions."),
    Document(page_content="Amber's PMEMD module is optimized for parallel execution on HPC clusters.")
]

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

chroma_client = Chroma.from_documents(
    fake_docs,
    embedding_model,
    persist_directory="./chromadb_fake",
    collection_name="amber_docs"
)
chroma_client.persist()

/tmp/ipython-input-4063542672.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/tmp/ipython-i

In [ ]:
def retrieve_context(query, chroma_client, top_k=5):
    results = chroma_client.similarity_search(query, k=top_k)
    context = "\n".join([r.page_content for r in results])
    return context

In [ ]:
def build_prompt(query, context):
    prompt = f"""You are Amber Support Assistant, an expert in Amber molecular dynamics software.

Context:
{context}

Question:
{query}

Answer the question clearly and concisely, using a step-by-step explanation when helpful."""
    return prompt


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import torch

model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to("cpu")

def generate_answer(prompt, max_new_tokens=300):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def rag_pipeline(user_query):
    context = retrieve_context(user_query, chroma_client)
    prompt = build_prompt(user_query, context)
    print("Prompt:\n", prompt)
    answer = generate_answer(prompt)
    return answer

In [ ]:
query = "How does Amber handle time-series anomaly detection?"
print("Running RAG pipeline...")
response = rag_pipeline(query)
print("\nAMBER Support:\n", response)